In [1]:
!pip install finnhub-python

In [10]:
import yfinance as yf
import boto3
import json
import time
from datetime import datetime, timedelta
from botocore.client import Config
import finnhub # <-- Thêm thư viện mới

# ==========================================
# CẤU HÌNH MINIO (Giữ nguyên của bạn)
# ==========================================
MINIO_ENDPOINT = "http://minio:9000"
ACCESS_KEY = "dataNLPmining-lab"
SECRET_KEY = "dataNLPmining-lab"
BUCKET_NAME = "raw-financial-data" 

s3_client = boto3.client('s3', endpoint_url=MINIO_ENDPOINT, aws_access_key_id=ACCESS_KEY, aws_secret_access_key=SECRET_KEY, config=Config(signature_version='s3v4', s3={'addressing_style': 'path'}), region_name='us-east-1')

# ==========================================
# CẤU HÌNH FINNHUB API
# ==========================================
FINNHUB_API_KEY = "d8p78f9r01qp954vlhggd8p78f9r01qp954vlhh0"
finnhub_client = finnhub.Client(api_key=FINNHUB_API_KEY)

# Hàm upload giữ nguyên
def upload_to_minio(data, folder, ticker):
    date_path = datetime.now().strftime("%Y/%m/%d")
    object_key = f"raw_zone_finnhub/{folder}/{date_path}/{ticker}.json" 
    json_bytes = json.dumps(data, ensure_ascii=False, default=str).encode('utf-8')
    try:
        s3_client.put_object(Bucket=BUCKET_NAME, Key=object_key, Body=json_bytes, ContentType='application/json')
        print(f"   -> ✅ Thành công lưu tại: s3://{BUCKET_NAME}/{object_key}")
    except Exception as e:
        print(f"   -> ❌ Lỗi upload lên MinIO: {e}")

# Hàm cào giá cổ phiếu (vẫn dùng yfinance vì nó làm việc này rất tốt)
def crawl_stock_price(ticker_symbol):
    print(f"⏳ Đang cào dữ liệu GIÁ cho mã {ticker_symbol}...")
    ticker = yf.Ticker(ticker_symbol)
    df = ticker.history(period="1y", interval="1d")
    if df.empty:
        return
    df.reset_index(inplace=True)
    df['Date'] = df['Date'].dt.strftime('%Y-%m-%d')
    upload_to_minio(data=df.to_dict(orient="records"), folder="market_data", ticker=ticker_symbol)

# ==========================================
# HÀM MỚI: CÀO TIN TỨC 1 NĂM BẰNG FINNHUB
# ==========================================
def crawl_stock_news_finnhub(ticker_symbol):
    print(f"📰 Đang cào dữ liệu TIN TỨC (1 năm) cho mã {ticker_symbol}...")
    
    end_date = datetime.now()
    start_date = end_date - timedelta(days=365) # Lùi về đúng 365 ngày
    
    all_news = []
    current_end = end_date
    
    # Quét lùi từng cụm 30 ngày để lấy sạch dữ liệu
    while current_end > start_date:
        current_start = current_end - timedelta(days=30)
        if current_start < start_date:
            current_start = start_date
            
        _from = current_start.strftime("%Y-%m-%d")
        _to = current_end.strftime("%Y-%m-%d")
        
        try:
            # Gọi API của Finnhub
            news_chunk = finnhub_client.company_news(ticker_symbol, _from=_from, to=_to)
            all_news.extend(news_chunk)
        except Exception as e:
            print(f"   -> ❌ Lỗi gọi API Finnhub cho đoạn {_from} đến {_to}: {e}")
            
        current_end = current_start
        time.sleep(1.5) # Rất quan trọng: Gói Free của Finnhub giới hạn 60 calls/phút
        
    if not all_news:
        print(f"❌ Không tìm thấy tin tức nào cho {ticker_symbol} trong năm qua.")
        return
        
    print(f"   -> Đã lấy được {len(all_news)} bài báo cho {ticker_symbol}.")
    upload_to_minio(data=all_news, folder="news_data_finnhub", ticker=ticker_symbol)



In [11]:
import pandas as pd
import requests

def get_sp500_tickers():
    print("⏳ Đang tải danh sách 500 mã cổ phiếu S&P 500 từ Wikipedia...")
    url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
    
    # 1. Tạo Header giả làm trình duyệt Chrome trên Windows
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    
    # 2. Tải mã nguồn trang web về bằng thư viện requests
    response = requests.get(url, headers=headers)
    
    # 3. Đưa mã HTML thô vào cho Pandas đọc
    tables = pd.read_html(response.text)
    df = tables[0]
    
    # Trích xuất cột Symbol (Mã cổ phiếu) và chuyển thành list
    tickers = df['Symbol'].tolist()
    
    # Xử lý các mã có dấu chấm (ví dụ BRK.B thành BRK-B)
    tickers = [ticker.replace('.', '-') for ticker in tickers]
    return tickers
if __name__ == "__main__":
    print(f"=== BẮT ĐẦU ĐẨY DỮ LIỆU LÊN BUCKET: {BUCKET_NAME} ===")
    
    # Lấy danh sách 500 mã (bạn có thể cắt lát [:50] nếu chỉ muốn chạy thử 50 mã đầu)
    danh_sach_ma = get_sp500_tickers()
    print(f"Tổng cộng có {len(danh_sach_ma)} mã cổ phiếu sẽ được cào.")
    
    for i, ma in enumerate(danh_sach_ma):
        print(f"\n--- Đang xử lý mã thứ {i+1}/{len(danh_sach_ma)}: {ma} ---")
        try:
            # 1. Cào giá
            crawl_stock_price(ma)
            time.sleep(1) 
            
            # 2. Cào tin tức
            crawl_stock_news_finnhub(ma)
            
            # Ngủ 12 giây sau mỗi mã để đảm bảo 1 phút chỉ gọi tối đa 5 mã (Tránh vượt ngưỡng 60 calls/min)
            print("   -> ⏳ Đang đợi 12 giây để tránh bị khóa API...")
            time.sleep(12) 
            
        except finnhub.FinnhubAPIException as e:
            if 'rate limit' in str(e).lower() or '429' in str(e):
                print("   -> ⚠️ Chạm ngưỡng API! Tạm dừng hệ thống 60 giây...")
                time.sleep(60)
                # Bạn có thể cân nhắc code thêm logic retry (thử lại) mã hiện tại ở đây
            else:
                print(f"   -> ❌ Lỗi API không xác định với mã {ma}: {e}")
        except Exception as e:
            print(f"   -> ❌ Lỗi hệ thống với mã {ma}: {e}")
            continue # Lỗi mã này thì bỏ qua chạy mã tiếp theo
            
    print("\n🎉 HOÀN TẤT CÀO TOÀN BỘ DỮ LIỆU!")

=== BẮT ĐẦU ĐẨY DỮ LIỆU LÊN BUCKET: raw-financial-data ===
⏳ Đang tải danh sách 500 mã cổ phiếu S&P 500 từ Wikipedia...


/tmp/ipykernel_17042/493286632.py:17: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text)


Tổng cộng có 503 mã cổ phiếu sẽ được cào.

--- Đang xử lý mã thứ 1/503: MMM ---
⏳ Đang cào dữ liệu GIÁ cho mã MMM...
   -> ✅ Thành công lưu tại: s3://raw-financial-data/raw_zone_finnhub/market_data/2026/06/17/MMM.json
📰 Đang cào dữ liệu TIN TỨC (1 năm) cho mã MMM...
   -> Đã lấy được 876 bài báo cho MMM.
   -> ✅ Thành công lưu tại: s3://raw-financial-data/raw_zone_finnhub/news_data_finnhub/2026/06/17/MMM.json
   -> ⏳ Đang đợi 12 giây để tránh bị khóa API...

--- Đang xử lý mã thứ 2/503: AOS ---
⏳ Đang cào dữ liệu GIÁ cho mã AOS...
   -> ✅ Thành công lưu tại: s3://raw-financial-data/raw_zone_finnhub/market_data/2026/06/17/AOS.json
📰 Đang cào dữ liệu TIN TỨC (1 năm) cho mã AOS...
   -> Đã lấy được 373 bài báo cho AOS.
   -> ✅ Thành công lưu tại: s3://raw-financial-data/raw_zone_finnhub/news_data_finnhub/2026/06/17/AOS.json
   -> ⏳ Đang đợi 12 giây để tránh bị khóa API...

--- Đang xử lý mã thứ 3/503: ABT ---
⏳ Đang cào dữ liệu GIÁ cho mã ABT...
   -> ✅ Thành công lưu tại: s3://raw-financ

Failed to get ticker 'APH' reason: Failed to perform, curl: (6) Connection closed abruptly. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.
Failed to get ticker 'ADI' reason: Failed to perform, curl: (6) Could not resolve host: query2.finance.yahoo.com. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.
Failed to get ticker 'AON' reason: Failed to perform, curl: (6) Could not resolve host: query2.finance.yahoo.com. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.
$AON: possibly delisted; no price data found  (period=1y)



--- Đang xử lý mã thứ 34/503: APH ---
⏳ Đang cào dữ liệu GIÁ cho mã APH...
   -> ❌ Lỗi hệ thống với mã APH: Failed to perform, curl: (6) Could not resolve host: query2.finance.yahoo.com. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.

--- Đang xử lý mã thứ 35/503: ADI ---
⏳ Đang cào dữ liệu GIÁ cho mã ADI...
   -> ❌ Lỗi hệ thống với mã ADI: Failed to perform, curl: (6) Could not resolve host: query2.finance.yahoo.com. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.

--- Đang xử lý mã thứ 36/503: AON ---
⏳ Đang cào dữ liệu GIÁ cho mã AON...
📰 Đang cào dữ liệu TIN TỨC (1 năm) cho mã AON...
   -> ❌ Lỗi gọi API Finnhub cho đoạn 2026-05-18 đến 2026-06-17: HTTPSConnectionPool(host='api.finnhub.io', port=443): Max retries exceeded with url: /api/v1//company-news?token=d8p78f9r01qp954vlhggd8p78f9r01qp954vlhh0&symbol=AON&from=2026-05-18&to=2026-06-17 (Caused by NameResolutionError("HTTPSConnection(host='api.finnhub.io', port=443): Failed to 

...

📰 Đang cào dữ liệu TIN TỨC (1 năm) cho mã IDXX...
   -> Đã lấy được 517 bài báo cho IDXX.
   -> ✅ Thành công lưu tại: s3://raw-financial-data/raw_zone_finnhub/news_data_finnhub/2026/06/17/IDXX.json
   -> ⏳ Đang đợi 12 giây để tránh bị khóa API...

--- Đang xử lý mã thứ 249/503: ITW ---
⏳ Đang cào dữ liệu GIÁ cho mã ITW...
   -> ✅ Thành công lưu tại: s3://raw-financial-data/raw_zone_finnhub/market_data/2026/06/17/ITW.json
📰 Đang cào dữ liệu TIN TỨC (1 năm) cho mã ITW...
   -> Đã lấy được 488 bài báo cho ITW.
   -> ✅ Thành công lưu tại: s3://raw-financial-data/raw_zone_finnhub/news_data_finnhub/2026/06/17/ITW.json
   -> ⏳ Đang đợi 12 giây để tránh bị khóa API...

--- Đang xử lý mã thứ 250/503: INCY ---
⏳ Đang cào dữ liệu GIÁ cho mã INCY...
   -> ✅ Thành công lưu tại: s3://raw-financial-data/raw_zone_finnhub/market_data/2026/06/17/INCY.json
📰 Đang cào dữ liệu TIN TỨC (1 năm) cho mã INCY...
   -> Đã lấy được 649 bài báo cho INCY.
   -> ✅ Thành công lưu tại: s3://raw-financial-data/raw_zone

Failed to get ticker 'LRCX' reason: Failed to perform, curl: (6) Connection closed abruptly. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.
$LRCX: possibly delisted; no price data found  (period=1y)



--- Đang xử lý mã thứ 284/503: LRCX ---
⏳ Đang cào dữ liệu GIÁ cho mã LRCX...
📰 Đang cào dữ liệu TIN TỨC (1 năm) cho mã LRCX...
   -> ❌ Lỗi gọi API Finnhub cho đoạn 2026-05-18 đến 2026-06-17: HTTPSConnectionPool(host='api.finnhub.io', port=443): Max retries exceeded with url: /api/v1//company-news?token=d8p78f9r01qp954vlhggd8p78f9r01qp954vlhh0&symbol=LRCX&from=2026-05-18&to=2026-06-17 (Caused by NameResolutionError("HTTPSConnection(host='api.finnhub.io', port=443): Failed to resolve 'api.finnhub.io' ([Errno -2] Name or service not known)"))
   -> ❌ Lỗi gọi API Finnhub cho đoạn 2026-03-19 đến 2026-04-18: HTTPSConnectionPool(host='api.finnhub.io', port=443): Read timed out. (read timeout=10)
   -> ❌ Lỗi gọi API Finnhub cho đoạn 2026-02-17 đến 2026-03-19: HTTPSConnectionPool(host='api.finnhub.io', port=443): Max retries exceeded with url: /api/v1//company-news?token=d8p78f9r01qp954vlhggd8p78f9r01qp954vlhh0&symbol=LRCX&from=2026-02-17&to=2026-03-19 (Caused by NameResolutionError("HTTPSCon

Failed to get ticker 'RTX' reason: Failed to perform, curl: (6) Connection closed abruptly. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.


   -> ✅ Thành công lưu tại: s3://raw-financial-data/raw_zone_finnhub/market_data/2026/06/17/RTX.json
📰 Đang cào dữ liệu TIN TỨC (1 năm) cho mã RTX...
   -> Đã lấy được 2042 bài báo cho RTX.
   -> ✅ Thành công lưu tại: s3://raw-financial-data/raw_zone_finnhub/news_data_finnhub/2026/06/17/RTX.json
   -> ⏳ Đang đợi 12 giây để tránh bị khóa API...

--- Đang xử lý mã thứ 390/503: O ---
⏳ Đang cào dữ liệu GIÁ cho mã O...
   -> ✅ Thành công lưu tại: s3://raw-financial-data/raw_zone_finnhub/market_data/2026/06/17/O.json
📰 Đang cào dữ liệu TIN TỨC (1 năm) cho mã O...
   -> Đã lấy được 1722 bài báo cho O.
   -> ✅ Thành công lưu tại: s3://raw-financial-data/raw_zone_finnhub/news_data_finnhub/2026/06/17/O.json
   -> ⏳ Đang đợi 12 giây để tránh bị khóa API...

--- Đang xử lý mã thứ 391/503: REG ---
⏳ Đang cào dữ liệu GIÁ cho mã REG...
   -> ✅ Thành công lưu tại: s3://raw-financial-data/raw_zone_finnhub/market_data/2026/06/17/REG.json
📰 Đang cào dữ liệu TIN TỨC (1 năm) cho mã REG...
   -> Đã lấy đượ

In [4]:
!pip install lxml


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 9.9 MB/s  0:00:00m eta 0:00:01
